# Where to Privatize? — DP Placement in PEFT: Results Analysis

**BERT-base + AG News | DP placements: no_dp, adapter_only, head_adapter, partial_backbone, last_layer, full_dp | ε ∈ {1.0, 8.0}, δ=1e-5**

This notebook loads all experiment results and generates paper-ready figures and tables.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path

# ── Academic plot style ──────────────────────────────────────────────────────
sns.set_style("whitegrid")
sns.set_context("paper", font_scale=1.4)
plt.rcParams.update({
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "font.family": "sans-serif",
    "axes.spines.top": False,
    "axes.spines.right": False,
})

# ── Consistent palette & labels ───────────────────────────────────────────────
PLACEMENT_ORDER = ["no_dp", "adapter_only", "head_adapter", "partial_backbone", "last_layer", "full_dp"]
LABELS = {
    "no_dp":            "No DP (Baseline)",
    "adapter_only":     "Adapter-Only DP",
    "head_adapter":     "Head+Adapter DP",
    "partial_backbone": "Partial Backbone DP",
    "last_layer":       "Last-Layer DP",
    "full_dp":          "Full-Model DP",
}
COLORS = {
    "no_dp":            "#555555",
    "adapter_only":     "#2ca02c",
    "head_adapter":     "#1f77b4",
    "partial_backbone": "#ff7f0e",
    "last_layer":       "#9467bd",
    "full_dp":          "#d62728",
}

FIGS = Path("../results/figures")
FIGS.mkdir(parents=True, exist_ok=True)
print("Figures will be saved to:", FIGS.resolve())

## Load Training Results

In [ ]:
BASE   = Path("../runpod_results")
REM    = BASE / "remaining results"
LAST   = Path("../last runs")
MIA_D  = Path("../mia_results")

# Canonical source for each experiment (priority: fixed run > remaining > base)
TRAIN_SOURCES = {
    "no_dp_eps8.0":            BASE  / "bert_agnews_no_dp_eps8.0.json",
    "no_dp_eps1.0":            BASE  / "bert_agnews_no_dp_eps1.0.json",
    "adapter_only_eps8.0":     BASE  / "bert_agnews_adapter_only_eps8.0.json",
    "adapter_only_eps1.0":     BASE  / "bert_agnews_adapter_only_eps1.0.json",
    "head_adapter_eps8.0":     BASE  / "bert_agnews_head_adapter_eps8.0.json",
    "head_adapter_eps1.0":     BASE  / "bert_agnews_head_adapter_eps1.0.json",
    "last_layer_eps8.0":       BASE  / "bert_agnews_last_layer_eps8.0.json",
    "last_layer_eps1.0":       BASE  / "bert_agnews_last_layer_eps1.0.json",
    "partial_backbone_eps8.0": BASE  / "bert_agnews_partial_backbone_eps8.0.json",
    "partial_backbone_eps1.0": REM   / "bert_agnews_partial_backbone_eps1.0.json",
    "full_dp_eps1.0":          LAST  / "bert_agnews_full_dp_eps1.0.json",
}

train_data = {}
for key, path in TRAIN_SOURCES.items():
    if path.exists():
        with open(path) as f:
            d = json.load(f)
        placement, eps = key.rsplit("_eps", 1)
        d["placement"] = placement
        d["epsilon"]   = float(eps)
        d["key"]       = key
        train_data[key] = d
    else:
        print(f"WARNING — missing: {path}")

rows = []
for k, d in train_data.items():
    rows.append({
        "key":             k,
        "placement":       d["placement"],
        "epsilon":         d["epsilon"],
        "accuracy":        d.get("final_accuracy"),
        "f1":              d.get("final_f1"),
        "loss":            d.get("final_loss"),
        "epoch_time":      d.get("avg_epoch_time"),
        "throughput":      d.get("avg_throughput"),
        "grad_var":        d.get("grad_norm_variance"),
        "loss_osc":        d.get("loss_oscillation"),
        "acc_curve":       d.get("accuracy_curve", []),
        "loss_curve":      d.get("convergence_curve", []),
    })

train_df = pd.DataFrame(rows)
print(f"Loaded {len(train_df)} training results")
print(train_df[["placement","epsilon","accuracy","f1","throughput"]].to_string(index=False))

## Load MIA Results

In [ ]:
mia_rows = []
for path in sorted(MIA_D.glob("bert_agnews_*_mia.json")):
    stem = path.stem.replace("bert_agnews_", "").replace("_mia", "")
    placement, eps = stem.rsplit("_eps", 1)
    with open(path) as f:
        d = json.load(f)
    ta = d.get("threshold_attack", {})
    mia_rows.append({
        "key":        stem,
        "placement":  placement,
        "epsilon":    float(eps),
        "auc":        float(ta.get("auc", np.nan)),
        "advantage":  float(ta.get("advantage", np.nan)),
        "tr_loss":    float(ta.get("train_loss_mean", np.nan)),
        "te_loss":    float(ta.get("test_loss_mean", np.nan)),
    })

mia_df = pd.DataFrame(mia_rows)
mia_df["loss_gap"] = mia_df["te_loss"] - mia_df["tr_loss"]
print(f"Loaded {len(mia_df)} MIA results")
print(mia_df[["placement","epsilon","auc","advantage","loss_gap"]].to_string(index=False))

## Figure 1 — Accuracy Comparison Across DP Placements

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))

placements  = PLACEMENT_ORDER
n           = len(placements)
bar_w       = 0.32
x           = np.arange(n)

# no_dp has one accuracy value used for both ε
def get_acc(p, eps):
    row = train_df[(train_df.placement == p) & (train_df.epsilon == eps)]
    if row.empty:
        # fall back to the other epsilon for no_dp
        row = train_df[train_df.placement == p]
    return float(row.iloc[0]["accuracy"]) if not row.empty else np.nan

acc8 = [get_acc(p, 8.0) for p in placements]
acc1 = [get_acc(p, 1.0) for p in placements]

bars8 = ax.bar(x - bar_w/2, acc8, bar_w, label="ε = 8.0",
               color=[COLORS[p] for p in placements], alpha=0.95, edgecolor="white", linewidth=0.5)
bars1 = ax.bar(x + bar_w/2, acc1, bar_w, label="ε = 1.0",
               color=[COLORS[p] for p in placements], alpha=0.55, edgecolor="white", linewidth=0.5,
               hatch="//")

# Annotate bars
for bar, val in zip(list(bars8) + list(bars1), acc8 + acc1):
    if not np.isnan(val):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                f"{val:.3f}", ha="center", va="bottom", fontsize=7.5, rotation=90)

ax.axhline(y=acc8[0], color=COLORS["no_dp"], linestyle="--", linewidth=1.2,
           alpha=0.6, label=f"No-DP ceiling ({acc8[0]:.3f})")

ax.set_xticks(x)
ax.set_xticklabels([LABELS[p] for p in placements], rotation=20, ha="right", fontsize=11)
ax.set_ylabel("Test Accuracy", fontsize=12)
ax.set_title("Classification Accuracy by DP Placement Strategy\n(BERT-base, AG News, 20 epochs)", fontsize=13, fontweight="bold")
ax.set_ylim(0.75, 0.97)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))

solid  = mpatches.Patch(facecolor="gray", alpha=0.95, label="ε = 8.0 (loose privacy)")
hatched = mpatches.Patch(facecolor="gray", alpha=0.55, hatch="//", label="ε = 1.0 (tight privacy)")
ax.legend(handles=[solid, hatched,
          mpatches.Patch(facecolor=COLORS["no_dp"], linestyle="--", label=f"No-DP ceiling")],
          fontsize=10, loc="lower right")

plt.tight_layout()
plt.savefig(FIGS / "fig1_accuracy_comparison.pdf")
plt.savefig(FIGS / "fig1_accuracy_comparison.png")
plt.show()
print("Saved fig1_accuracy_comparison")

## Figure 2 — Membership Inference Attack Results

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))

for ax, metric, ylabel, title, ref, reflabel in [
    (ax1, "auc",       "MIA Attack AUC",       "MIA AUC (threshold attack)",       0.50, "Random guess (0.50)"),
    (ax2, "advantage", "MIA Attack Advantage",  "MIA Advantage (max TPR − FPR)",    0.00, "Ideal DP (0.00)"),
]:
    bar_w = 0.32
    ps    = [p for p in PLACEMENT_ORDER if p in mia_df.placement.values]
    x     = np.arange(len(ps))

    def get_mia(p, eps, m=metric):
        row = mia_df[(mia_df.placement == p) & (mia_df.epsilon == eps)]
        if row.empty:
            row = mia_df[mia_df.placement == p]
        return float(row.iloc[0][m]) if not row.empty else np.nan

    v8 = [get_mia(p, 8.0) for p in ps]
    v1 = [get_mia(p, 1.0) for p in ps]

    b8 = ax.bar(x - bar_w/2, v8, bar_w, color=[COLORS[p] for p in ps],
                alpha=0.95, edgecolor="white", label="ε = 8.0")
    b1 = ax.bar(x + bar_w/2, v1, bar_w, color=[COLORS[p] for p in ps],
                alpha=0.55, edgecolor="white", hatch="//", label="ε = 1.0")

    ax.axhline(ref, color="black", linestyle="--", linewidth=1.3, label=reflabel)
    ax.set_xticks(x)
    ax.set_xticklabels([LABELS[p] for p in ps], rotation=22, ha="right", fontsize=9.5)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_title(title, fontsize=12, fontweight="bold")

    # annotate
    for bar, val in zip(list(b8)+list(b1), v8+v1):
        if not np.isnan(val):
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.0003,
                    f"{val:.4f}", ha="center", va="bottom", fontsize=7, rotation=90)

ax1.set_ylim(0.497, 0.535)
ax2.set_ylim(-0.002, 0.088)
ax1.legend(fontsize=9, loc="upper right")
ax2.legend(fontsize=9, loc="upper right")

fig.suptitle("Empirical Privacy: Membership Inference Attack Results\n(BERT-base, AG News)",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(FIGS / "fig2_mia_results.pdf")
plt.savefig(FIGS / "fig2_mia_results.png")
plt.show()
print("Saved fig2_mia_results")

## Figure 3 — Train-Test Loss Gap (MIA Mechanism)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))

ps   = [p for p in PLACEMENT_ORDER if p in mia_df.placement.values]
x    = np.arange(len(ps))
bw   = 0.32

def get_loss(p, col, eps=8.0):
    row = mia_df[(mia_df.placement == p) & (mia_df.epsilon == eps)]
    if row.empty:
        row = mia_df[mia_df.placement == p]
    return float(row.iloc[0][col]) if not row.empty else np.nan

tr8 = [get_loss(p, "tr_loss") for p in ps]
te8 = [get_loss(p, "te_loss") for p in ps]

ax.bar(x - bw/2, tr8, bw, label="Train loss (mean)", color=[COLORS[p] for p in ps], alpha=0.95, edgecolor="white")
ax.bar(x + bw/2, te8, bw, label="Test loss (mean)",  color=[COLORS[p] for p in ps], alpha=0.50, edgecolor="white", hatch="//")

# draw gap arrows
for i, (tr, te) in enumerate(zip(tr8, te8)):
    if not (np.isnan(tr) or np.isnan(te)) and abs(te - tr) > 0.015:
        ax.annotate("", xy=(i + bw/2, te), xytext=(i + bw/2, tr),
                    arrowprops=dict(arrowstyle="<->", color="black", lw=1.2))
        ax.text(i + bw/2 + 0.05, (tr+te)/2, f"Δ={te-tr:.3f}", fontsize=8, va="center")

ax.set_xticks(x)
ax.set_xticklabels([LABELS[p] for p in ps], rotation=20, ha="right", fontsize=10)
ax.set_ylabel("Per-sample cross-entropy loss", fontsize=11)
ax.set_title("Train vs Test Loss Gap Across Placements (ε = 8.0)\n"
             "Large gap → MIA exploitable; near-zero gap → DP working",
             fontsize=12, fontweight="bold")

solid   = mpatches.Patch(facecolor="gray", alpha=0.95, label="Train loss (mean)")
hatched = mpatches.Patch(facecolor="gray", alpha=0.50, hatch="//", label="Test loss (mean)")
ax.legend(handles=[solid, hatched], fontsize=10)

plt.tight_layout()
plt.savefig(FIGS / "fig3_loss_gap.pdf")
plt.savefig(FIGS / "fig3_loss_gap.png")
plt.show()
print("Saved fig3_loss_gap")

## Figure 4 — Training Convergence (Loss & Accuracy Curves, ε = 1.0)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))

for p in PLACEMENT_ORDER:
    row = train_df[(train_df.placement == p) & (train_df.epsilon == 1.0)]
    if row.empty:
        row = train_df[train_df.placement == p]
    if row.empty:
        continue
    r       = row.iloc[0]
    lc      = r["loss_curve"]
    ac      = r["acc_curve"]
    epochs  = range(1, len(lc) + 1)
    lbl     = LABELS[p]
    c       = COLORS[p]
    lw      = 2.5 if p in ("adapter_only", "no_dp") else 1.8
    ls      = "--" if p == "no_dp" else "-"

    ax1.plot(epochs, lc, color=c, linewidth=lw, linestyle=ls, label=lbl)
    if ac:
        ax2.plot(range(1, len(ac)+1), [v*100 for v in ac],
                 color=c, linewidth=lw, linestyle=ls, label=lbl)

ax1.set_xlabel("Epoch", fontsize=11); ax1.set_ylabel("Training Loss", fontsize=11)
ax1.set_title("Loss Convergence (ε = 1.0)", fontsize=12, fontweight="bold")
ax1.legend(fontsize=9, loc="upper right")

ax2.set_xlabel("Epoch", fontsize=11); ax2.set_ylabel("Test Accuracy (%)", fontsize=11)
ax2.set_title("Accuracy Curve (ε = 1.0)", fontsize=12, fontweight="bold")
ax2.legend(fontsize=9, loc="lower right")
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0f}%"))

fig.suptitle("Training Dynamics Across DP Placements (BERT-base, AG News)",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(FIGS / "fig4_convergence.pdf")
plt.savefig(FIGS / "fig4_convergence.png")
plt.show()
print("Saved fig4_convergence")

## Figure 5 — Training Stability & Systems Efficiency

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

metrics = [
    ("grad_var",   "Gradient Norm Variance",        "Stability: Grad Norm Variance"),
    ("epoch_time", "Avg Epoch Time (s)",             "Training Cost: Epoch Time"),
    ("throughput", "Throughput (samples / sec)",     "Training Cost: Throughput"),
]

for ax, (col, ylabel, title) in zip(axes, metrics):
    vals, colors, labels = [], [], []
    for p in PLACEMENT_ORDER:
        row = train_df[(train_df.placement == p) & (train_df.epsilon == 1.0)]
        if row.empty:
            row = train_df[train_df.placement == p]
        if not row.empty and not np.isnan(row.iloc[0][col]):
            vals.append(float(row.iloc[0][col]))
            colors.append(COLORS[p])
            labels.append(LABELS[p])

    x = np.arange(len(labels))
    bars = ax.bar(x, vals, color=colors, alpha=0.88, edgecolor="white", linewidth=0.5)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=28, ha="right", fontsize=8.5)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.set_title(title, fontsize=11, fontweight="bold")

    for bar, val in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()*1.01,
                f"{val:.0f}" if val > 1 else f"{val:.4f}",
                ha="center", va="bottom", fontsize=8)

plt.suptitle("Training Stability & Efficiency by DP Placement", fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(FIGS / "fig5_stability_efficiency.pdf")
plt.savefig(FIGS / "fig5_stability_efficiency.png")
plt.show()
print("Saved fig5_stability_efficiency")

## Figure 6 — Utility vs Privacy Scatter (Key Summary Plot)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

merged = pd.merge(
    train_df[["placement","epsilon","accuracy"]],
    mia_df[["placement","epsilon","advantage"]],
    on=["placement","epsilon"], how="inner"
)

for _, row in merged.iterrows():
    p   = row["placement"]
    eps = row["epsilon"]
    ax.scatter(row["advantage"], row["accuracy"]*100,
               color=COLORS[p], s=150 if eps == 8.0 else 80,
               marker="o" if eps == 8.0 else "s",
               zorder=3, edgecolors="white", linewidths=0.7)
    offset = (0.0005, 0.15)
    ax.annotate(f"{LABELS[p]}\n(ε={'8' if eps==8.0 else '1'})",
                xy=(row["advantage"], row["accuracy"]*100),
                xytext=(row["advantage"]+offset[0], row["accuracy"]*100+offset[1]),
                fontsize=7.5, color=COLORS[p])

ax.set_xlabel("MIA Attack Advantage  →  (higher = less private)", fontsize=11)
ax.set_ylabel("Test Accuracy (%)", fontsize=11)
ax.set_title("Privacy-Utility Frontier: Accuracy vs MIA Advantage\n"
             "●=ε 8.0  ■=ε 1.0   Bottom-left = ideal (high privacy, high utility)",
             fontsize=11, fontweight="bold")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0f}%"))

# ideal region annotation
ax.annotate("← Better privacy\n↑ Better utility\n(ideal corner)", xy=(0.002, 89),
            fontsize=9, color="green",
            bbox=dict(boxstyle="round,pad=0.3", facecolor="lightgreen", alpha=0.3))

ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIGS / "fig6_utility_privacy_frontier.pdf")
plt.savefig(FIGS / "fig6_utility_privacy_frontier.png")
plt.show()
print("Saved fig6_utility_privacy_frontier")

## Tables — LaTeX Summary for Report

In [ ]:
nodp_acc = float(train_df[train_df.placement=="no_dp"].iloc[0]["accuracy"])

summary_rows = []
for p in PLACEMENT_ORDER:
    for eps in [8.0, 1.0]:
        tr = train_df[(train_df.placement==p) & (train_df.epsilon==eps)]
        mi = mia_df[(mia_df.placement==p) & (mia_df.epsilon==eps)]

        acc  = float(tr.iloc[0]["accuracy"])    if not tr.empty else None
        et   = float(tr.iloc[0]["epoch_time"])   if not tr.empty else None
        gv   = float(tr.iloc[0]["grad_var"])     if not tr.empty else None
        auc  = float(mi.iloc[0]["auc"])          if not mi.empty else None
        adv  = float(mi.iloc[0]["advantage"])    if not mi.empty else None
        drop = (nodp_acc - acc)                  if acc is not None else None

        summary_rows.append({
            "Placement":    LABELS[p],
            "eps":          "—" if p=="no_dp" else f"{eps:.0f}",
            "Accuracy":     f"{acc:.4f}"    if acc  is not None else "—",
            "D_NoDp":       f"-{drop:.3f}" if drop is not None else "—",
            "Epoch_s":      f"{et:.0f}"     if et   is not None else "—",
            "Grad_Var":     f"{gv:.4f}"     if gv   is not None else "—",
            "MIA_AUC":      f"{auc:.4f}"    if auc  is not None else "—",
            "MIA_Adv":      f"{adv:.4f}"    if adv  is not None else "—",
        })
        if p == "no_dp":
            break

summary_df = pd.DataFrame(summary_rows)
print("=" * 95)
print("SUMMARY TABLE")
print("=" * 95)
print(summary_df.rename(columns={"D_NoDp":"Δ No-DP","Epoch_s":"Epoch(s)","Grad_Var":"GradVar",
                                  "MIA_AUC":"MIA AUC","MIA_Adv":"MIA Adv"}).to_string(index=False))

# ── Manual LaTeX table (no jinja2 needed) ──────────────────────────────────
cols = ["Placement","eps","Accuracy","D_NoDp","Epoch_s","Grad_Var","MIA_AUC","MIA_Adv"]
col_labels = ["Placement", r"$\varepsilon$", "Accuracy", r"$\Delta$ No-DP",
              "Epoch (s)", "Grad. Var.", "MIA AUC", "MIA Adv."]

lines = [
    r"\begin{table}[h]",
    r"\centering",
    r"\caption{Results: BERT-base on AG News. $\delta=10^{-5}$ for all DP runs.}",
    r"\label{tab:main_results}",
    r"\begin{tabular}{llcccccc}",
    r"\hline",
    " & ".join(col_labels) + r" \\",
    r"\hline",
]
for _, row in summary_df.iterrows():
    lines.append(" & ".join(str(row[c]) for c in cols) + r" \\")
lines += [r"\hline", r"\end{tabular}", r"\end{table}"]

latex_str = "\n".join(lines)
print("\n\n=== LaTeX Table ===\n")
print(latex_str)

# Save to file for easy copy-paste
(Path("../results") / "summary_table.tex").write_text(latex_str)
print("\nSaved to results/summary_table.tex")

## Key Quantitative Findings

In [ ]:
def get_acc(p, eps):
    r = train_df[(train_df.placement==p) & (train_df.epsilon==eps)]
    return float(r.iloc[0]["accuracy"]) if not r.empty else None

def get_adv(p, eps):
    r = mia_df[(mia_df.placement==p) & (mia_df.epsilon==eps)]
    return float(r.iloc[0]["advantage"]) if not r.empty else None

nodp_acc = get_acc("no_dp", 8.0)
ao8      = get_acc("adapter_only", 8.0)
ao1      = get_acc("adapter_only", 1.0)
fd1      = get_acc("full_dp", 1.0)
ll8      = get_acc("last_layer", 8.0)
fd_et    = float(train_df[(train_df.placement=="full_dp")&(train_df.epsilon==1.0)].iloc[0]["epoch_time"])
ao_et    = float(train_df[(train_df.placement=="adapter_only")&(train_df.epsilon==8.0)].iloc[0]["epoch_time"])
nodp_adv = get_adv("no_dp",       8.0)
ao8_adv  = get_adv("adapter_only", 8.0)

print("Key Quantitative Findings")
print("=" * 60)
print(f"1.  No-DP baseline accuracy             : {nodp_acc:.2%}")
print(f"2.  Adapter-Only @ ε=8 accuracy         : {ao8:.2%}  (drop: {nodp_acc-ao8:.2%})")
print(f"3.  Adapter-Only @ ε=1 accuracy         : {ao1:.2%}  (drop: {nodp_acc-ao1:.2%})")
print(f"4.  ε sensitivity (adapter-only)         : {ao8-ao1:.2%}  ← near zero")
print(f"5.  Full-DP @ ε=1 accuracy              : {fd1:.2%}  (drop: {nodp_acc-fd1:.2%})")
print(f"6.  Adapter-Only vs Full-DP gap @ ε=1   : {ao1-fd1:.2%}  ← core thesis")
print(f"7.  No-DP MIA advantage                 : {nodp_adv:.4f}")
print(f"8.  Adapter-Only MIA advantage @ ε=8    : {ao8_adv:.4f}  ({nodp_adv/ao8_adv:.1f}× reduction)")
print(f"9.  Full-DP epoch time vs Adapter-Only  : {fd_et:.0f}s vs {ao_et:.0f}s  ({fd_et/ao_et:.1f}× slower)")
print(f"10. Last-Layer accuracy @ ε=8           : {ll8:.2%}  (drop: {nodp_acc-ll8:.2%})")

## Key Findings

1. **Adapter-Only DP** achieves the best privacy-utility tradeoff
2. Applying DP to fewer parameters preserves more utility
3. All placements provide similar empirical privacy protection
4. Training stability varies significantly across placements